# eFrog RL Post-Training — REINFORCE walkthrough

This notebook mirrors `train_rl.py`. It loads the ONNX classifier as a trainable
policy, runs a REINFORCE update on user observations that have feedback, and
exports a new ONNX model.

**RL formulation** — policy `π_θ = softmax(classifier logits)`, state = mel
spectrogram, action = sampled class, reward = `+1` if the sampled class equals
the user-confirmed `species_name` else `-1`, objective = REINFORCE with a
running-mean baseline and an entropy bonus.

Run the cells top to bottom. With no real data, the *synthetic* path at the
bottom exercises the whole pipeline.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import torch

# Pipeline pieces live in train_rl.py / make_synthetic_data.py in this folder.
import make_synthetic_data as synth
import train_rl as rl
from labels import LABEL_CLASSES, N_CLASSES

print('classes:', N_CLASSES)
print(torch.__version__)

## 1. Inputs

Point `MODEL_PATH` at your trained ONNX classifier (e.g.
`../artifacts/frog_classifier.onnx`) and `CSV_PATH` at a Supabase
`Version_1.observations` CSV export.

If you don't have real data handy, run the synthetic cell below first; it writes
files into `_smoke/` and sets these paths for you.

In [ ]:
# --- Synthetic inputs (skip if you have real data) ---
work = Path('_smoke'); work.mkdir(exist_ok=True)
MODEL_PATH = synth.make_synthetic_model(work / 'synthetic_classifier.onnx')
CSV_PATH = synth.make_synthetic_csv(work / 'synthetic_observations.csv', n_rows=48, seed=42)
OUT_PATH = work / 'frog_classifier_rl.onnx'
print('model:', MODEL_PATH)
print('csv  :', CSV_PATH)

## 2. Load & filter observations

Keeps only rows with `included_feedback = true`, a valid 10048-float mel, and a
`species_name` in the 19-class list. The label is `species_name` → class index.

In [ ]:
states, labels, stats = rl.load_observations(Path(CSV_PATH))
print('states:', states.shape, 'labels:', labels.shape)
for k, v in stats.items():
    print(f'  {k:22s}: {v}')

## 3. Load the policy network (ONNX → trainable torch module)

`onnx2torch_compat` (imported inside `load_policy`) registers the opset-18
reduce converters that `onnx2torch` 1.5.x lacks.

In [ ]:
model = rl.load_policy(Path(MODEL_PATH))
with torch.no_grad():
    logits = rl._policy_logits(model, torch.from_numpy(states[:2]))
print('logits shape:', tuple(logits.shape))  # expect (2, 19)

## 4. REINFORCE training

Sample an action per state, reward `±1`, advantage = reward − running-mean
baseline, loss `= -(adv)·logπ(a|s) - β·H`. Watch `mean_reward` / `greedy_acc`
per epoch. (More epochs / higher lr on the synthetic data clearly show learning.)

In [ ]:
device = torch.device('cpu')
model = rl.train(
    model, states, labels,
    epochs=20, lr=1e-3, batch_size=16, entropy_coef=0.01, seed=42, device=device,
)

## 5. Export & verify the new ONNX

Same signature as the input: `input [1,1,64,157]` → `output [1,19]`, opset 18.

In [ ]:
rl.export_onnx(model, Path(OUT_PATH))
rl.verify_onnx(Path(OUT_PATH))
print('wrote', OUT_PATH)